# Multi-Layer Perceptron (MLP) / Neural Network

**INDE 577 / CMOR 438 — Rice University**  
**Instructor:** Randy R. Davila, PhD

---

## Overview

A **Multi-Layer Perceptron (MLP)** is a fully-connected feedforward neural network. By stacking layers of neurons with non-linear activation functions, MLPs can approximate any continuous function (Universal Approximation Theorem).

## Mathematical Background

### Forward Pass

For a network with $L$ layers, the forward pass computes:

$$\mathbf{a}^{(0)} = \mathbf{x} \quad \text{(input)}$$

$$\mathbf{z}^{(l)} = \mathbf{W}^{(l)} \mathbf{a}^{(l-1)} + \mathbf{b}^{(l)}$$

$$\mathbf{a}^{(l)} = f\left(\mathbf{z}^{(l)}\right)$$

where $f$ is the activation function (ReLU, sigmoid, tanh, etc.).

### Activation Functions

$$\text{ReLU}(z) = \max(0, z)$$

$$\text{Sigmoid}(z) = \frac{1}{1 + e^{-z}}$$

$$\text{Tanh}(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$

### Backpropagation

The gradient of the loss $\mathcal{L}$ w.r.t. parameters is computed via the chain rule:

$$\boldsymbol{\delta}^{(L)} = \nabla_{\mathbf{a}^{(L)}} \mathcal{L} \odot f'\left(\mathbf{z}^{(L)}\right)$$

$$\boldsymbol{\delta}^{(l)} = \left(\mathbf{W}^{(l+1)T} \boldsymbol{\delta}^{(l+1)}\right) \odot f'\left(\mathbf{z}^{(l)}\right)$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}^{(l)}} = \boldsymbol{\delta}^{(l)} \left(\mathbf{a}^{(l-1)}\right)^T, \quad \frac{\partial \mathcal{L}}{\partial \mathbf{b}^{(l)}} = \boldsymbol{\delta}^{(l)}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_circles, make_moons
from sklearn.neural_network import MLPClassifier as SklearnMLP
import warnings
warnings.filterwarnings('ignore')

from rice_ml import MLP
from rice_ml.processing.preprocessing import StandardScaler, train_test_split
from rice_ml.processing.metrics import accuracy_score, classification_report

print("Libraries loaded!")
np.random.seed(42)

## 1. XOR Problem — Where Perceptron Failed

In [ ]:
# XOR
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

# MLP with 1 hidden layer of 4 neurons
mlp_xor = MLP(hidden_layers=[4], activation='tanh', learning_rate=0.5,
              n_iterations=5000, task='classification', random_state=42)
mlp_xor.fit(X_xor, y_xor)

y_pred_xor = mlp_xor.predict(X_xor)
print("XOR solved by MLP:")
for xi, yt, yp in zip(X_xor, y_xor, y_pred_xor):
    mark = '✓' if yt == yp else '✗'
    print(f"  {xi.astype(int)} → {yp}  {mark}")
print(f"Accuracy: {accuracy_score(y_xor, y_pred_xor):.2f}")

## 2. Non-Linear Datasets (Circles & Moons)

In [ ]:
datasets = [
    ('Circles', *make_circles(n_samples=300, noise=0.1, factor=0.3, random_state=42)),
    ('Moons', *make_moons(n_samples=300, noise=0.15, random_state=42))
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (name, X_d, y_d) in zip(axes, datasets):
    scaler = StandardScaler()
    X_d_s = scaler.fit_transform(X_d)
    X_tr, X_te, y_tr, y_te = train_test_split(X_d_s, y_d, test_size=0.2, random_state=42)
    
    mlp = MLP(hidden_layers=[16, 8], activation='relu', learning_rate=0.01,
              n_iterations=2000, task='classification', random_state=42)
    mlp.fit(X_tr, y_tr)
    
    x_min, x_max = X_d_s[:, 0].min() - 0.5, X_d_s[:, 0].max() + 0.5
    y_min, y_max = X_d_s[:, 1].min() - 0.5, X_d_s[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = mlp.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.scatter(X_d_s[y_d == 0, 0], X_d_s[y_d == 0, 1], c='blue', s=25, alpha=0.7, edgecolors='k', lw=0.3)
    ax.scatter(X_d_s[y_d == 1, 0], X_d_s[y_d == 1, 1], c='red', s=25, alpha=0.7, edgecolors='k', lw=0.3)
    acc = accuracy_score(y_te, mlp.predict(X_te))
    ax.set_title(f'MLP on {name} (test acc={acc:.2f})', fontsize=13, fontweight='bold')
    ax.set_xlabel('Feature 1', fontsize=11)
    ax.set_ylabel('Feature 2', fontsize=11)

plt.suptitle('MLP Learns Non-Linear Decision Boundaries', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('mlp_nonlinear.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Breast Cancer Classification

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
target_names = data.target_names

scaler = StandardScaler()
X_s = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_s, y, test_size=0.2, random_state=42)

# rice_ml MLP
mlp_bc = MLP(hidden_layers=[64, 32], activation='relu', learning_rate=0.001,
             n_iterations=1000, task='classification', lambda_reg=0.001, random_state=42)
mlp_bc.fit(X_train, y_train)
y_pred = mlp_bc.predict(X_test)

# sklearn MLP
sk_mlp = SklearnMLP(hidden_layer_sizes=(64, 32), activation='relu', max_iter=1000, random_state=42)
sk_mlp.fit(X_train, y_train)

our_acc = accuracy_score(y_test, y_pred)
sk_acc = accuracy_score(y_test, sk_mlp.predict(X_test))

print("=== Breast Cancer Classification ===")
print(classification_report(y_test, y_pred, target_names=list(target_names)))
print(f"rice_ml MLP accuracy: {our_acc:.4f}")
print(f"sklearn MLP accuracy:  {sk_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax = axes[0]
ax.plot(mlp_bc.loss_history_, color='crimson', linewidth=2)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('MLP Training Loss (Breast Cancer)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_facecolor('#f8f8f8')

# Accuracy during training
ax2 = axes[1]
if hasattr(mlp_bc, 'accuracy_history_') and mlp_bc.accuracy_history_:
    ax2.plot(mlp_bc.accuracy_history_, color='steelblue', linewidth=2)
    ax2.set_ylabel('Training Accuracy', fontsize=11)
else:
    # Plot architecture visualization
    layer_sizes = [X_train.shape[1]] + [64, 32] + [1]
    layer_names = ['Input\n(30)', 'Hidden\n(64)', 'Hidden\n(32)', 'Output\n(1)']
    x_pos = np.arange(len(layer_sizes))
    ax2.bar(x_pos, layer_sizes, color=['#3498db', '#e74c3c', '#e74c3c', '#2ecc71'],
            edgecolor='k', linewidth=0.7, width=0.5)
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(layer_names, fontsize=10)
    ax2.set_ylabel('Number of Neurons', fontsize=11)
ax2.set_xlabel('Layer', fontsize=11)
ax2.set_title('Network Architecture', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('mlp_cancer.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Effect of Architecture

In [ ]:
architectures = [
    ([8], 'Shallow (8)'),
    ([32], 'Shallow (32)'),
    ([64, 32], 'Deep (64-32)'),
    ([128, 64, 32], 'Deep (128-64-32)'),
]

print(f"{'Architecture':<20} {'Test Acc':>10} {'Train Acc':>10}")
print("-" * 42)
for layers, name in architectures:
    m = MLP(hidden_layers=layers, activation='relu', learning_rate=0.001,
            n_iterations=500, task='classification', random_state=42)
    m.fit(X_train, y_train)
    tr_acc = accuracy_score(y_train, m.predict(X_train))
    te_acc = accuracy_score(y_test, m.predict(X_test))
    print(f"{name:<20} {te_acc:>10.4f} {tr_acc:>10.4f}")

## Summary

| Property | Value |
|---|---|
| Type | Fully-connected feedforward neural network |
| Training | Backpropagation + gradient descent (Adam/SGD) |
| Activation | ReLU, Sigmoid, Tanh |
| Regularization | L2 weight decay |
| Capability | Universal function approximator |
| Dataset | Breast Cancer (569 × 30) |

**Key Takeaways:**
- MLPs overcome the XOR limitation by learning **non-linear decision boundaries**
- More layers = more expressive capacity, but also more risk of overfitting
- ReLU is the most popular activation for hidden layers (avoids vanishing gradients)
- Feature standardization is critical for neural network training